In [9]:
import os
print("Current folder:", os.getcwd())
print("Files in raw:", os.listdir("../data/raw"))

Current folder: D:\DATA ANLAYTICS\python
Files in raw: ['amazon_sale_report_v1.csv', 'amazon_sale_report_v2.csv', 'Sale Report.csv']


In [10]:
import pandas as pd

df = pd.read_csv("../data/raw/amazon_sale_report_v1.csv", low_memory=False)
df = df.drop(columns=['index', 'Unnamed: 22'])
print(df.shape)

(128975, 22)


In [11]:
df.columns = (df.columns.str.strip().str.lower()
              .str.replace('-', '_').str.replace(' ', '_'))
print(list(df.columns))

['order_id', 'date', 'status', 'fulfilment', 'sales_channel', 'ship_service_level', 'style', 'sku', 'category', 'size', 'asin', 'courier_status', 'qty', 'currency', 'amount', 'ship_city', 'ship_state', 'ship_postal_code', 'ship_country', 'promotion_ids', 'b2b', 'fulfilled_by']


In [12]:
df['date'] = pd.to_datetime(df['date'], format='%m-%d-%y')
df['ship_postal_code'] = df['ship_postal_code'].astype('Int64').astype('string')
df['month'] = df['date'].dt.to_period('M').astype(str)
df['week'] = df['date'].dt.to_period('W').astype(str)

In [13]:
df['ship_state'] = df['ship_state'].str.strip().str.title()

state_fix = {
    'Ar': 'Arunachal Pradesh', 'Nl': 'Nagaland', 'Pb': 'Punjab', 'Rj': 'Rajasthan',
    'Orissa': 'Odisha', 'Pondicherry': 'Puducherry', 'New Delhi': 'Delhi',
    'Rajshthan': 'Rajasthan', 'Rajsthan': 'Rajasthan',
    'Punjab/Mohali/Zirakpur': 'Punjab',
    'Andaman & Nicobar': 'Andaman & Nicobar Islands',
    'Dadra And Nagar': 'Dadra and Nagar Haveli',
}
df['ship_state'] = df['ship_state'].replace(state_fix)

print(df['ship_state'].nunique())
print(sorted(df['ship_state'].dropna().unique()))

37
['Andaman & Nicobar Islands', 'Andhra Pradesh', 'Apo', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra and Nagar Haveli', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu & Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Ladakh', 'Lakshadweep', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']


In [14]:
df['ship_state'] = df['ship_state'].str.strip().str.title()

state_fix = {
    'Ar': 'Arunachal Pradesh', 'Nl': 'Nagaland', 'Pb': 'Punjab', 'Rj': 'Rajasthan',
    'Orissa': 'Odisha', 'Pondicherry': 'Puducherry', 'New Delhi': 'Delhi',
    'Rajshthan': 'Rajasthan', 'Rajsthan': 'Rajasthan',
    'Punjab/Mohali/Zirakpur': 'Punjab',
    'Andaman & Nicobar': 'Andaman & Nicobar Islands',
    'Dadra And Nagar': 'Dadra and Nagar Haveli',
}
df['ship_state'] = df['ship_state'].replace(state_fix)

print(df['ship_state'].nunique())
print(sorted(df['ship_state'].dropna().unique()))

37
['Andaman & Nicobar Islands', 'Andhra Pradesh', 'Apo', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra And Nagar Haveli', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu & Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Ladakh', 'Lakshadweep', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']


In [15]:
def group_status(s):
    if s == 'Cancelled':
        return 'Cancelled'
    if s in ['Shipped - Returned to Seller', 'Shipped - Returning to Seller', 'Shipped - Rejected by Buyer']:
        return 'Returned'
    if s == 'Shipped - Delivered to Buyer':
        return 'Delivered'
    if s in ['Shipped - Lost in Transit', 'Shipped - Damaged']:
        return 'Lost/Damaged'
    return 'Shipped / In progress'

df['status_group'] = df['status'].apply(group_status)
df['is_cancelled'] = (df['status_group'] == 'Cancelled').astype(int)
df['is_returned'] = (df['status_group'] == 'Returned').astype(int)

print(df['status_group'].value_counts())
print("Cancellation rate:", round(df['is_cancelled'].mean() * 100, 1), "%")

status_group
Shipped / In progress    79759
Delivered                28769
Cancelled                18332
Returned                  2109
Lost/Damaged                 6
Name: count, dtype: int64
Cancellation rate: 14.2 %


In [17]:
df.to_csv("../data/clean/amazon_clean.csv", index=False)
print("Saved:", df.shape)

Saved: (128975, 27)


In [21]:
order_level = df.groupby('order_id')['is_cancelled'].max()
print("Orders:", len(order_level))
print("Order-level cancellation rate:", round(order_level.mean() * 100, 1), "%")

Orders: 120378
Order-level cancellation rate: 14.3 %


In [19]:
print(df[df['qty'] == 0]['status_group'].value_counts())

status_group
Cancelled                12701
Shipped / In progress       95
Delivered                    8
Returned                     3
Name: count, dtype: int64


In [20]:
print(df[df['ship_state'] == 'Apo'][['ship_city', 'ship_postal_code', 'status_group']])
print("Missing state:", df['ship_state'].isnull().sum())

      ship_city ship_postal_code status_group
45187       APO           959121    Cancelled
Missing state: 33


In [22]:
df.to_csv("../data/clean/amazon_clean.csv", index=False)
print("Saved:", df.shape)

Saved: (128975, 27)


In [23]:
df.to_csv("../data/clean/amazon_clean.csv", index=False)

In [24]:
import sqlite3
conn = sqlite3.connect("../sql/amazon.db")
df.to_sql("orders", conn, if_exists="replace", index=False)
print(pd.read_sql("SELECT COUNT(*) AS total_rows FROM orders", conn))

   total_rows
0      128975


In [25]:
query = """
SELECT ship_state,
       COUNT(*) AS order_lines,
       SUM(is_cancelled) AS cancelled,
       ROUND(100.0 * SUM(is_cancelled) / COUNT(*), 1) AS cancel_rate_pct
FROM orders
WHERE ship_state IS NOT NULL AND ship_state <> 'Apo'
GROUP BY ship_state
HAVING COUNT(*) >= 500
ORDER BY cancel_rate_pct DESC
"""
pd.read_sql(query, conn)

,ship_state,order_lines,cancelled,cancel_rate_pct
0,Himachal Pradesh,788,146,18.5
1,Kerala,6585,1175,17.8
2,Jammu & Kashmir,702,119,17.0
3,Andhra Pradesh,5430,892,16.4
4,Odisha,2139,346,16.2
5,Bihar,2114,341,16.1
6,Jharkhand,1456,231,15.9
7,Uttarakhand,1553,234,15.1
8,Uttar Pradesh,10638,1604,15.1
9,Madhya Pradesh,2529,380,15.0
